# Scraper lịch sự cho một trang danh mục sản phẩm

[books.toscrape.com](https://books.toscrape.com) là một nhà sách **giả** do Zyte (Scrapinghub) dựng để luyện web
scraping: 1 000 cuốn sách chia thành 50 trang danh mục, mỗi trang 20 cuốn. Tên sách lấy từ sách thật, nhưng **giá,
số sao và tình trạng kho là dữ liệu giả**. Website cho phép crawl, nên đây là nơi luyện kỹ thuật mà không vi phạm điều
khoản của ai.

**Bối cảnh.** Team pricing của một nhà sách online muốn theo dõi giá của đối thủ. Trước khi viết scraper cho website
thật (và hỏi pháp chế xem có được không), bạn dựng một bản mẫu trên sandbox: lấy tên, giá, số sao, tình trạng kho,
đường dẫn của mọi cuốn sách, crawl **lịch sự** (kiểm `robots.txt`, nghỉ giữa các request, có giới hạn số trang, không
lặp vô hạn), rồi trả lời một câu hỏi thử: sách được đánh giá cao có đắt hơn không?

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` gọi website thật (khoảng 50 request, nghỉ 0.5 giây giữa các request); khi chấm chúng bị bỏ qua,
  test dùng hai trang đã lưu trong `tests/fixtures/`.
- Chấm: `uv run pytest tracks/data-analyst/modules/05-data-collection/exercises/basic/03-books-scraper -q`

In [ ]:
import re
import time
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
# explore: client có User-Agent tự giới thiệu, đọc robots.txt trước khi crawl
import httpx

START = "https://books.toscrape.com/catalogue/page-1.html"
client = httpx.Client(
    headers={"User-Agent": "da-learning-course/1.0 (scraping exercise; sandbox books.toscrape.com)"},
    timeout=20,
    follow_redirects=True,
)
robots = client.get("https://books.toscrape.com/robots.txt")
print("robots.txt:", robots.status_code)
robots_txt = robots.text if robots.status_code == 200 else None


def fetch(url: str) -> bytes:
    resp = client.get(url)
    resp.raise_for_status()
    return resp.content


page1 = fetch(START)
print(page1[:120])

## 1. Đọc một trang danh mục

Mỗi cuốn sách là một `<article class="product_pod">`. Mở `tests/fixtures/page-1.html` (hoặc "View source" trên
trình duyệt) để xem cấu trúc, rồi viết:

`parse_price(text) -> float`: lấy số đầu tiên trong chuỗi giá, bỏ dấu phẩy ngăn cách hàng nghìn. Phải chịu được
`"£51.77"`, `"Â£51.77"` (chữ `£` bị giải mã sai thành `Â£`, xem cell explore phía dưới) và khoảng trắng thừa; không
có số thì `ValueError`.

`parse_listing(html, page_url) -> list[dict]`: `html` là `bytes` hoặc `str`. Mỗi cuốn một dict, đúng thứ tự trên
trang, khoá:

- `title`: tiêu đề **đầy đủ** trong thuộc tính `title` của thẻ `<h3><a>` (chữ hiển thị bị cắt thành `...`);
- `price`: `parse_price` của `<p class="price_color">`;
- `rating`: số sao 1–5, đọc từ class thứ hai của `<p class="star-rating Three">`;
- `in_stock`: `True` nếu ô `<p class="availability">` chứa `In stock`;
- `url`: đường dẫn tuyệt đối của trang sách, ghép `href` tương đối với `page_url` bằng `urljoin`.

`next_page(html, page_url) -> str | None`: URL tuyệt đối của nút `<li class="next"><a>`, hoặc `None` ở trang cuối.

In [ ]:
RATINGS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


def parse_price(text: str) -> float:
    """Số đầu tiên trong chuỗi giá, bỏ dấu phẩy hàng nghìn."""
    # TODO: re.search một số dạng 51.77; không thấy thì ValueError
    raise NotImplementedError


def parse_listing(html, page_url: str) -> list[dict]:
    """title, price, rating, in_stock, url của mọi sách trên trang danh mục."""
    # TODO: BeautifulSoup(html, "lxml"), duyệt "article.product_pod"
    raise NotImplementedError


def next_page(html, page_url: str) -> str | None:
    """URL tuyệt đối của trang kế tiếp, None nếu là trang cuối."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: trang đầu, và vì sao "£" có thể thành "Â£"
books = parse_listing(page1, START)
print(len(books), books[0])
print("next:", next_page(page1, START))

import requests
r = requests.get(START, headers={"User-Agent": client.headers["User-Agent"]}, timeout=20)
print("Content-Type:", r.headers.get("Content-Type"), "| requests đoán:", r.encoding)
print("r.text chứa 'Â£'?", "Â£" in r.text, "| r.content.decode('utf-8') chứa 'Â£'?", "Â£" in r.content.decode("utf-8"))

## 2. robots.txt và crawl lịch sự

`robots_allows(robots_txt, url, agent="*") -> bool`: `robots_txt` là nội dung tệp `robots.txt` (chuỗi), hoặc `None`
khi website không có tệp đó (HTTP 404; theo RFC 9309, lỗi 4xx nghĩa là không có giới hạn). Dùng
`urllib.robotparser.RobotFileParser` (gọi `.parse(dòng)` thay vì `.read()` để không gọi mạng).

`crawl(fetch, start_url, max_pages=50, delay=1.0, sleep=time.sleep) -> pd.DataFrame`:

- `fetch(url) -> bytes` là hàm tải trang (test truyền hàm giả, notebook truyền hàm dùng `httpx`);
- bắt đầu ở `start_url`, theo `next_page` tới khi hết trang, **hoặc** đã tải `max_pages` trang, **hoặc** URL kế tiếp đã
  tải rồi (chống lặp vô hạn);
- gọi `sleep(delay)` **giữa** hai request liên tiếp (không nghỉ trước request đầu);
- trả bảng cột `title, price, rating, in_stock, url`, bỏ dòng trùng `url`, index 0..n-1.

In [ ]:
COLUMNS = ["title", "price", "rating", "in_stock", "url"]


def robots_allows(robots_txt: str | None, url: str, agent: str = "*") -> bool:
    """robots.txt có cho agent tải url không; None (không có tệp) nghĩa là cho."""
    # TODO
    raise NotImplementedError


def crawl(fetch, start_url: str, max_pages: int = 50, delay: float = 1.0, sleep=time.sleep) -> pd.DataFrame:
    """Theo nút next, nghỉ delay giây giữa các request, dừng ở trang cuối / max_pages / URL đã tải."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: crawl cả danh mục (50 request, nghỉ 0.5 giây: khoảng 1-2 phút)
assert robots_allows(robots_txt, START, agent=client.headers["User-Agent"])
started = time.monotonic()
catalog = crawl(fetch, START, max_pages=60, delay=0.5)
print(f"{len(catalog)} sách, {catalog.url.nunique()} URL, {catalog.title.nunique()} tiêu đề khác nhau, "
      f"{time.monotonic() - started:.0f} giây")
print(catalog[catalog.title.duplicated(keep=False)][["title", "price", "url"]])

In [ ]:
# explore: sách nhiều sao có đắt hơn không?
import matplotlib.pyplot as plt

by_rating = catalog.groupby("rating").price.agg(["count", "mean", "median"]).round(2)
print(by_rating)
print("tương quan giá - số sao:", round(catalog.price.corr(catalog.rating), 3))
print("tỉ lệ còn hàng:", catalog.in_stock.mean())

fig, ax = plt.subplots(figsize=(7, 4))
catalog.boxplot(column="price", by="rating", ax=ax, grid=False)
ax.set_title("Giá gần như không đổi theo số sao (dữ liệu sandbox)", loc="left")
fig.suptitle("")
ax.set_xlabel("số sao")
ax.set_ylabel("giá (GBP)")
plt.show()

## 3. Đọc kết quả, rồi làm trắc nghiệm

1. Bạn đã làm gì để crawl "lịch sự"? Nếu website thật có `robots.txt` cấm `/catalogue/`, bạn làm gì?
2. Vì sao khoá chống trùng là `url` chứ không phải `title`?
3. "Sách nhiều sao có đắt hơn không?" Số trên sandbox nói gì, và vì sao **không** được kết luận gì về thị trường sách từ
   số này?
4. Trước khi chạy scraper này trên website thật của đối thủ, cần hỏi những gì (điều khoản sử dụng, dữ liệu cá nhân, tần
   suất, có API hay feed chính thức không)?

Xong thì mở app, làm phần **trắc nghiệm** của bài.